# Phase 1: Data Ingest

### Imports & Configs

In [0]:
import requests
import os
from pyspark.sql.functions import date_trunc, col, lit, count
import time 

In [0]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
TMP_PATH = "/Volumes/workspace/default/staging/tlc_temp.parquet" 
END_YEAR  = 2025
END_MONTH = 12

VEHICLE_TYPES = {
    "yellow": {
        "url_prefix":    "yellow_tripdata",
        "datetime_col":  "tpep_pickup_datetime",
        "start_year":    2019,
        "start_month":   1
    },
    "green": {
        "url_prefix":    "green_tripdata",
        "datetime_col":  "lpep_pickup_datetime",
        "start_year":    2019,
        "start_month":   1
    },
    "hvfhv": {
        "url_prefix":    "fhvhv_tripdata",
        "datetime_col":  "pickup_datetime",
        "start_year":    2019,
        "start_month":   2
    }
}

### Create Tables

In [0]:
# hourly demand
# USING DELTA tells databricks to create a delta table

spark.sql("""
    CREATE TABLE IF NOT EXISTS hourly_demand (
        hour         TIMESTAMP,
        vehicle_type STRING,
        trip_count   LONG
    )
    USING DELTA
""")

# ingest log

spark.sql("""
    CREATE TABLE IF NOT EXISTS ingest_log (
        vehicle_type  STRING,
        year          INT,
        month         INT,
        row_count     LONG,
        status        STRING,
        processed_at  TIMESTAMP
    )
    USING DELTA
""")

# staging 

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.staging")

print("Tables ready.")

Tables ready.


### Ingest Loop


In [0]:
# Load checkpoint upfront — which files are already done
# ingest_log is the source of truth, done is just the in-memory working copy of it
# using a set because is it faster for checking if r in something

done = set()
try:
    rows = spark.sql("SELECT vehicle_type, year, month FROM ingest_log WHERE status = 'success'").collect() # make a list of the rows in ingest_log where status = success
    for r in rows:
        done.add((r.vehicle_type, r.year, r.month)) # convert into tuple, and add to done
    print(f"Checkpoint loaded — {len(done)} files already done")
except:
    print("Starting fresh — no checkpoint found")

# Ingest loop
# loop through the vtype dicts
for vtype, config in VEHICLE_TYPES.items():
    for year in range(config["start_year"], END_YEAR + 1): # loop through the years 
        start_m = config["start_month"] if year == config["start_year"] else 1 # if this is the first year for this vehicle type, use the configured start month, otherwise start from January
        end_m   = END_MONTH if year == END_YEAR else 12

        for month in range(start_m, end_m + 1): # loop through the months 

            if (vtype, year, month) in done: # check if tuple is in done (the set of tuples), if it is continue
                print(f"  Skipping {vtype} {year}-{month:02d} — already done") 
                continue # stop here and jump to the next iteration of the loop  i.e. check the next tuple

            url = f"{BASE_URL}/{config['url_prefix']}_{year}-{month:02d}.parquet" # build the url 
            print(f"Processing {vtype} {year}-{month:02d}...")

            try:
                # Stream download to Volume staging area
                with requests.get(url, stream=True, timeout=600) as request: # stream TRUE tells requests to download in chunks
                    request.raise_for_status()
                    with open(TMP_PATH, "wb") as output_file: # open a file on disk
                        for chunk in request.iter_content(chunk_size=8 * 1024 * 1024):
                            output_file.write(chunk) # write the chunk to file 

                # Read parquet into df, then build hourly dataframe — 3 cols: hour (datetime truncated to hour), trip_count (frequency per hour), vehicle_type (fixed for this file)
                df = spark.read.parquet(TMP_PATH) # make dataframe from file         
                hourly = (
                    df
                    .select(date_trunc("hour", col(config["datetime_col"])).alias("hour")) # select the datetime col, truncated to hour, renamed to hour
                    .groupBy("hour")                        # group on the hour col
                    .agg(count("*").alias("trip_count"))    # count rows per hour and name as trip_count
                    .withColumn("vehicle_type", lit(vtype)) # add the vehicle_type col
                )

                # Get total trips
                # spark dataframe -> python list - Row(sum(trip_count)=482391) -> [ Row(sum(trip_count)=482391) ]
                # collect() takes the spark dataframe and returns a list of rows
                total_trips = hourly.agg({"trip_count": "sum"}).collect()[0][0]

                # Append to Delta table
                hourly.write.format("delta").mode("append").saveAsTable("hourly_demand")

                # Log success
                spark.sql(f"""
                    INSERT INTO ingest_log
                    VALUES ('{vtype}', {year}, {month}, {total_trips}, 'success', current_timestamp())
                """)

                # Update done
                done.add((vtype, year, month))
                print(f"  Done — {total_trips:,} trips")

            except Exception as error: # catch (intercept before it crashes) any error that occurs, whatever the cause
                print(f"  FAILED: {error}")
                spark.sql(f"""
                    INSERT INTO ingest_log
                    VALUES ('{vtype}', {year}, {month}, 0, 'failed', current_timestamp())
                """)
            # Delete the temp file
            finally:
                if os.path.exists(TMP_PATH):
                    os.remove(TMP_PATH)

            time.sleep(2)  # avoid rate limiting from TLC CDN

Checkpoint loaded — 55 files already done
  Skipping yellow 2019-01 — already done
  Skipping yellow 2019-02 — already done
  Skipping yellow 2019-03 — already done
  Skipping yellow 2019-04 — already done
  Skipping yellow 2019-05 — already done
  Skipping yellow 2019-06 — already done
  Skipping yellow 2019-07 — already done
  Skipping yellow 2019-08 — already done
  Skipping yellow 2019-09 — already done
  Skipping yellow 2019-10 — already done
  Skipping yellow 2019-11 — already done
  Skipping yellow 2019-12 — already done
  Skipping yellow 2020-01 — already done
  Skipping yellow 2020-02 — already done
  Skipping yellow 2020-03 — already done
  Skipping yellow 2020-04 — already done
  Skipping yellow 2020-05 — already done
  Skipping yellow 2020-06 — already done
  Skipping yellow 2020-07 — already done
  Skipping yellow 2020-08 — already done
  Skipping yellow 2020-09 — already done
  Skipping yellow 2020-10 — already done
  Skipping yellow 2020-11 — already done
  Skipping yell

### Test run

In [0]:
# Test cell
# Run this before the full ingest to verify the pipeline end-to-end.
# Safe to re-run — ingest_log checkpointing will skip it if already done.

vtype  = "yellow"
year   = 2024
month  = 1

cfg = VEHICLE_TYPES[vtype]
url = f"{BASE_URL}/{cfg['url_prefix']}_{year}-{month:02d}.parquet"

print(f"Testing: {vtype} {year}-{month:02d}")
print(f"URL: {url}")

try:
    # Download
    with requests.get(url, stream=True, timeout=600) as request:
        request.raise_for_status()
        with open(TMP_PATH, "wb") as output_file:
            for chunk in request.iter_content(chunk_size=8 * 1024 * 1024):
                output_file.write(chunk)
    print(f"  Downloaded: {os.path.getsize(TMP_PATH) / 1e6:.1f} MB")

    # Aggregate
    df = spark.read.parquet(TMP_PATH)
    hourly = (
        df
        .select(date_trunc("hour", col(cfg["datetime_col"])).alias("hour"))
        .groupBy("hour")
        .agg(count("*").alias("trip_count"))
        .withColumn("vehicle_type", lit(vtype))
    )
    total_trips = hourly.agg({"trip_count": "sum"}).collect()[0][0]
    print(f"  Hours: {hourly.count()}  |  Total trips: {total_trips:,}")

    # Append to Delta
    hourly.write.format("delta").mode("append").saveAsTable("hourly_demand")
    print(f"  Appended to hourly_demand")

    # Log success
    spark.sql(f"""
        INSERT INTO ingest_log
        VALUES ('{vtype}', {year}, {month}, {total_trips}, 'success', current_timestamp())
    """)
    print(f"  Logged to ingest_log")

except Exception as error:
    print(f"  FAILED: {error}")
    spark.sql(f"""
        INSERT INTO ingest_log
        VALUES ('{vtype}', {year}, {month}, 0, 'failed', current_timestamp())
    """)

finally:
    if os.path.exists(TMP_PATH):
        os.remove(TMP_PATH)
        print(f"  Temp file deleted")

# Verify
print("\n── hourly_demand sample ──")
spark.sql("SELECT * FROM hourly_demand ORDER BY hour LIMIT 5").show()

print("\n── ingest_log ──")
spark.sql("SELECT * FROM ingest_log").show()

Testing: yellow 2024-01
URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
  Downloaded: 50.0 MB
  Hours: 749  |  Total trips: 2,964,624
  Appended to hourly_demand
  Logged to ingest_log
  Temp file deleted

── hourly_demand sample ──
+-------------------+------------+----------+
|               hour|vehicle_type|trip_count|
+-------------------+------------+----------+
|2002-12-31 22:00:00|      yellow|         2|
|2009-01-01 00:00:00|      yellow|         1|
|2009-01-01 23:00:00|      yellow|         2|
|2023-12-31 23:00:00|      yellow|        10|
|2024-01-01 00:00:00|      yellow|      6596|
+-------------------+------------+----------+


── ingest_log ──
+------------+----+-----+---------+-------+--------------------+
|vehicle_type|year|month|row_count| status|        processed_at|
+------------+----+-----+---------+-------+--------------------+
|      yellow|2024|    1|  2964624|success|2026-07-12 17:38:...|
|      yellow|2024|    1|        0| f

## Data Quality Checks

#### 1. Row counts 
Total row counts per vehicle type, filtered to 2019-2025 to exclude bogus timestamps outside
the expected range. Green returned more rows than HVFHV, which is unexpected. HVFHV averages
around 26,000 trips per hour and should have at least one trip in every hour across the full
date range. Green averages around 70 trips per hour and will have gaps in quieter periods, so
it should have fewer distinct hours than HVFHV. This prompted a year-by-year breakdown.

In [0]:
spark.sql("""
        SELECT 
            vehicle_type,
            COUNT(*)
        FROM hourly_demand    
        WHERE hour >= '2019-01-01' AND hour < '2026-01-01'    
        GROUP BY vehicle_type
""").show()

+------------+--------+
|vehicle_type|COUNT(*)|
+------------+--------+
|       hvfhv|   60614|
|       green|   62122|
|      yellow|   63681|
+------------+--------+



Year-by-year row counts per vehicle type. Green and Yellow both exceed the maximum possible
hours per year (8,760 for standard years, 8,784 for leap years 2020 and 2024), confirming
duplicate (hour, vehicle_type) rows exist in both. HVFHV hits the maximum exactly each year.
The 2019 shortfall (8,015 vs 8,016 expected) reflects the missing January file and one hour
with zero trips recorded. Duplicate checks below investigate the cause for Green and Yellow.

**Note:** This cell was re-run after the deduplication fix was applied. Current output reflects the deduplicated table - Green and Yellow counts are now at or below the expected hours per year. The original finding (counts exceeding expected) is what prompted the duplicate checks below.

In [0]:
spark.sql("""
    SELECT
        vehicle_type,
        YEAR(hour) AS year,
        COUNT(*) AS row_count
    FROM hourly_demand
    WHERE hour >= '2019-01-01' AND hour < '2026-01-01'
    GROUP BY vehicle_type, YEAR(hour)
    ORDER BY vehicle_type, year
""").show()

+------------+----+---------+
|vehicle_type|year|row_count|
+------------+----+---------+
|       green|2019|     8760|
|       green|2020|     8781|
|       green|2021|     8754|
|       green|2022|     8759|
|       green|2023|     8759|
|       green|2024|     8779|
|       green|2025|     8754|
|       hvfhv|2019|     8015|
|       hvfhv|2020|     8775|
|       hvfhv|2021|     8760|
|       hvfhv|2022|     8760|
|       hvfhv|2023|     8760|
|       hvfhv|2024|     8784|
|       hvfhv|2025|     8760|
|      yellow|2019|     8759|
|      yellow|2020|     8783|
|      yellow|2021|     8759|
|      yellow|2022|     8759|
|      yellow|2023|     8759|
|      yellow|2024|     8783|
+------------+----+---------+
only showing top 20 rows


#### 2. Duplicates

Checked for duplicate `(hour, vehicle_type)` combinations - rows where the same vehicle type
has more than one trip count recorded for the same hour. Yellow and Green both had duplicates,
with some hours appearing up to 4 times. Note: this query returns
0 rows as the fix has already been applied. See the verification section below.

In [0]:
spark.sql("""
    SELECT
        hour,
        vehicle_type,
        COUNT(*) AS frequency_count
    FROM hourly_demand
    WHERE hour >= '2019-01-01' AND hour < '2026-01-01'
    GROUP BY hour, vehicle_type
    HAVING frequency_count > 1
    ORDER BY frequency_count DESC
""").show()

+----+------------+---------------+
|hour|vehicle_type|frequency_count|
+----+------------+---------------+
+----+------------+---------------+



Queried the ingest log to rule out pipeline re-runs as the cause. Returned 251 successes,
one per file and matching the expected file count exactly, with no file appearing more than
once. Re-run ruled out.

In [0]:
spark.sql("""
    SELECT 
        status, 
        COUNT(*) AS count
    FROM ingest_log
    GROUP BY status
""").show()

+-------+-----+
| status|count|
+-------+-----+
|success|  251|
| failed|  200|
+-------+-----+



In [0]:
spark.sql("""
    SELECT 
        vehicle_type, 
        year, 
        month, 
        COUNT(*) AS log_entries
    FROM ingest_log
    WHERE status = 'success'
    GROUP BY vehicle_type, year, month
    HAVING COUNT(*) > 1
    ORDER BY log_entries DESC
""").show()

+------------+----+-----+-----------+
|vehicle_type|year|month|log_entries|
+------------+----+-----+-----------+
+------------+----+-----+-----------+



Checked whether duplicate rows for the same `(hour, vehicle_type)` carry identical or different
trip counts. All duplicate groups had differing values, confirming the rows came from different
source files. Root cause: TLC Parquet files contain spillover trips from adjacent months.
Spillover ranged from single stray trips (`min_tc = 1`) up to ~241 trips for some hours. HVFHV
is unaffected, consistent with it being a newer and better-maintained dataset.

In [0]:
spark.sql("""
    SELECT
        hour,
        vehicle_type,
        COUNT(*)          AS row_count,
        MIN(trip_count)   AS min_tc,
        MAX(trip_count)   AS max_tc
    FROM hourly_demand
    WHERE hour >= '2019-01-01' AND hour < '2026-01-01' 
    GROUP BY hour, vehicle_type
    HAVING COUNT(*) > 1 AND MIN(trip_count) > 200
    ORDER BY row_count DESC
    LIMIT 20
""").show()

+----+------------+---------+------+------+
|hour|vehicle_type|row_count|min_tc|max_tc|
+----+------------+---------+------+------+
+----+------------+---------+------+------+



Collapsed `hourly_demand` to one row per `(hour, vehicle_type)` using `MAX(trip_count)`. MAX
was chosen because the spillover rows always carried the smaller count, so it preserves the
authoritative demand figure for the correct month and discards the spillover without
double-counting.

In [0]:
from pyspark.sql import functions as F

# make a new df called dedpued
deduped = (
    spark.table("hourly_demand")
    .groupBy("hour", "vehicle_type")
    .agg(F.max("trip_count").alias("trip_count")) # take the max trip count of any duplicate rows
)

deduped.write.format("delta").mode("overwrite").saveAsTable("hourly_demand") # write deduped over hourly_demand

Post-fix duplicate check returned 0 rows. Row counts now match expected hours per year for all
vehicle types.

In [0]:
spark.sql("""
    SELECT
        hour,
        vehicle_type,
        COUNT(*) AS frequency_count
    FROM hourly_demand
    WHERE hour >= '2019-01-01' AND hour < '2026-01-01'
    GROUP BY hour, vehicle_type
    HAVING frequency_count > 1
    ORDER BY frequency_count DESC
""").show()

+----+------------+---------------+
|hour|vehicle_type|frequency_count|
+----+------------+---------------+
+----+------------+---------------+



#### 3. Null check 

Null counts across all three columns in `hourly_demand`. All returned 0 — no missing values
in hour, vehicle_type, or trip_count.

In [0]:
spark.sql("""
        SELECT
            COUNT(*) - COUNT(hour) AS hour_nulls,
            COUNT(*) - COUNT(vehicle_type) AS vehicle_type_nulls,
            COUNT(*) - COUNT(trip_count) AS trip_count_nulls
        FROM hourly_demand       
""").show()

+----------+------------------+----------------+
|hour_nulls|vehicle_type_nulls|trip_count_nulls|
+----------+------------------+----------------+
|         0|                 0|               0|
+----------+------------------+----------------+



#### 4. Date Range 

Confirms the min and max `hour` per vehicle type and identifies any out-of-range timestamps in the source data.

Yellow and Green contain out-of-range timestamps: a small number of trip records in the NYC Taxi and Limousine Commission (TLC) source Parquet files carry corrupt or placeholder datetime values, producing rows with pickup times as early as 2001 and as late as 2098. This is a known TLC source data quality issue. HVFHV is clean, with min and max matching the expected range exactly (HVFHV reporting began February 2019).

**Handling:** All downstream queries filter `hour` to `2019-01-01` through `2025-12-31`. The out-of-range rows exist in `hourly_demand` but are excluded from every analysis, chart, and model. No fix to the table is required.

In [0]:
spark.sql("""
        SELECT 
            vehicle_type,
            MIN(hour) AS min_hour,
            MAX(hour) AS max_hour
        FROM hourly_demand
        GROUP BY vehicle_type   
""").show()

+------------+-------------------+-------------------+
|vehicle_type|           min_hour|           max_hour|
+------------+-------------------+-------------------+
|      yellow|2001-01-01 00:00:00|2098-09-11 02:00:00|
|       green|2008-10-21 15:00:00|2062-08-15 00:00:00|
|       hvfhv|2019-02-01 00:00:00|2025-12-31 23:00:00|
+------------+-------------------+-------------------+



#### 5. Continuity Check: distinct hour count vs expected

After deduplication, one row equals one distinct hour. This query counts rows per year per type and compares against the total hours in each calendar year to confirm no systematic gaps exist.

Expected hours per year:

| Year | Expected hours | Note |
|---|---|---|
| 2019 | 8,760 | HVFHV: 8,016 (reporting begins February 2019) |
| 2020 | 8,784 | Leap year |
| 2021 | 8,760 | |
| 2022 | 8,760 | |
| 2023 | 8,760 | |
| 2024 | 8,784 | Leap year |
| 2025 | 8,760 | |

All shortfalls are small. The largest is HVFHV 2020 (9 hours), consistent with the deepest COVID lockdown period in April 2020 when demand across all types dropped to near zero. Single-hour shortfalls in other years reflect isolated hours with no recorded trips. No systematic gaps were found.

HVFHV 2019 shows 8,015 against an expected 8,016 (reporting begins February 2019, giving 334 days). One hour in the Feb-Dec 2019 range had zero trips.

In [0]:
spark.sql("""
    SELECT
        vehicle_type,
        YEAR(hour) AS year,
        COUNT(*) AS row_count
    FROM hourly_demand
    WHERE hour >= '2019-01-01' AND hour < '2026-01-01'
    GROUP BY vehicle_type, YEAR(hour)
    ORDER BY vehicle_type, year
""").show(25)

+------------+----+---------+
|vehicle_type|year|row_count|
+------------+----+---------+
|       green|2019|     8760|
|       green|2020|     8781|
|       green|2021|     8754|
|       green|2022|     8759|
|       green|2023|     8759|
|       green|2024|     8779|
|       green|2025|     8754|
|       hvfhv|2019|     8015|
|       hvfhv|2020|     8775|
|       hvfhv|2021|     8760|
|       hvfhv|2022|     8760|
|       hvfhv|2023|     8760|
|       hvfhv|2024|     8784|
|       hvfhv|2025|     8760|
|      yellow|2019|     8759|
|      yellow|2020|     8783|
|      yellow|2021|     8759|
|      yellow|2022|     8759|
|      yellow|2023|     8759|
|      yellow|2024|     8783|
|      yellow|2025|     8759|
+------------+----+---------+



#### 6. Bogus Timestamps: count of rows outside 2019–2025

Counts rows in `hourly_demand` with an `hour` value outside the expected 2019-2025 range, confirming the extent of the corrupt timestamp issue identified in the date range check above.

Green has 66 bogus rows across 8 spurious years. Yellow has 241 bogus rows spread across years ranging from 2001 to 2098. HVFHV has none, consistent with the clean min/max seen in the date range check.

The bogus row counts are negligible relative to the total size of each type's series (tens of thousands of rows per type per year). All downstream queries filter `hour` to `2019-01-01` through `2025-12-31`, so these rows are excluded from every analysis, chart, and model.

In [0]:
spark.sql("""
    SELECT
        vehicle_type,
        YEAR(hour) AS year,
        COUNT(*) AS row_count
    FROM hourly_demand
    WHERE hour < '2019-01-01' OR hour >= '2026-01-01'
    GROUP BY vehicle_type, YEAR(hour)
    ORDER BY vehicle_type, year
""").show(100)

+------------+----+---------+
|vehicle_type|year|row_count|
+------------+----+---------+
|       green|2008|        7|
|       green|2009|       22|
|       green|2010|        4|
|       green|2018|       20|
|       green|2026|       10|
|       green|2035|        1|
|       green|2041|        1|
|       green|2062|        1|
|      yellow|2001|        5|
|      yellow|2002|      138|
|      yellow|2003|       12|
|      yellow|2004|        1|
|      yellow|2007|        1|
|      yellow|2008|       14|
|      yellow|2009|       26|
|      yellow|2010|        1|
|      yellow|2011|        2|
|      yellow|2012|        1|
|      yellow|2014|        1|
|      yellow|2015|        1|
|      yellow|2018|       19|
|      yellow|2026|        2|
|      yellow|2028|        1|
|      yellow|2029|        4|
|      yellow|2033|        2|
|      yellow|2038|        2|
|      yellow|2041|        1|
|      yellow|2058|        2|
|      yellow|2066|        1|
|      yellow|2070|        1|
|      yel